# Model Answer Evaluation Notebook

This notebook compares three AI model answers against correct reference answers.
It loads a CSV file, measures how good each answer is, and saves the results.

Input: A CSV file with these columns:
- id: a unique ID for each question
- correct_answer: the reference answer we compare against
- sources: the correct legal sources (paragraph references like § 5 Abs. 1)
- answer_inference: answer from the base inference model
- answer_sft: answer from the fine-tuned model
- answer_rag: answer from the retrieval-augmented model

Process:
1. Load and inspect the data
2. Clean and normalize the text
3. Compute scores for each answer: Exact Match, BLEU, ROUGE, Similarity, BERTScore
4. Check whether the model cited the correct legal sources
5. Summarize results per model and analyze errors

Output: Three CSV files saved to the output folder:
- detailed_scores.csv: one row per (question, model) pair with all scores
- main_result_table.csv: one row per model with average scores
- error_analysis.csv: breakdown of error types per model

## Step 1: Import Libraries

We load all the tools we will use in this notebook.
Each import has a short comment explaining what it is for.

In [ ]:
import os
import re
import difflib
import pandas as pd
# rouge_scorer measures how much two texts overlap (ROUGE metric)
from rouge_score import rouge_scorer
# sentence_bleu measures word overlap between two sentences (BLEU metric)
# SmoothingFunction prevents a score of zero for very short texts
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# bert_score measures meaning similarity using a language model
from bert_score import score as bert_score
# These two are needed to fix a known bug in the bert_score library
from transformers import BertTokenizer, BertTokenizerFast

## Step 2: Set File Paths

Here we define where to read the input file from and where to save the results.

In [ ]:
# Path to the input CSV file
input_csv = "./out_model_comp/total_answers.csv"

# Folder where all output files will be saved
output_folder = "out_model_comp"

# Create the folder if it does not already exist
os.makedirs(output_folder, exist_ok=True)

## Step 3: Load the Data

We read the CSV file into a dataframe, which is a table Python can work with.
Each row in the table is one question with the correct answer and three model answers.

In [ ]:
# Read the CSV file into a dataframe
df = pd.read_csv(input_csv)

# Replace any empty cells with an empty string so we do not get errors later
df = df.fillna("")

# Show the first few rows to check the data looks right
df.head()

## Step 4: Check That All Required Columns Are Present

Before computing any scores, we make sure all the expected columns exist in the file.
If one is missing, we stop and show a clear error message.

In [ ]:
# All column names we expect to find in the CSV
required_columns = ["id", "correct_answer", "sources", "answer_inference", "answer_sft", "answer_rag"]

# Check each column one by one
for column in required_columns:
    if column not in df.columns:
        raise ValueError("Missing column: " + column)

## Step 5: Set Up the Scoring Tools

We prepare the ROUGE scorer and the BLEU smoothing function before the main loop.
We also define which model columns we want to evaluate.

In [ ]:
# The ROUGE scorer checks word overlap between two texts
# rouge1 counts single-word matches, rougeL counts the longest matching sequence
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

# This smoothing avoids BLEU returning zero on very short texts
smoothing = SmoothingFunction().method1

# The three models we want to evaluate
# Each entry contains: (column name in the CSV, short label for the output)
models = [("answer_inference", "inference"), ("answer_sft", "sft"), ("answer_rag", "rag")]

## Step 6: Fix a Known Bug in the BERTScore Library

The current version of BERTScore has a small problem with how it prepares text for the BERT model.
The code below applies a fix for this issue.

In [ ]:
# This corrected function adds the special tokens that BERT needs around each text
# CLS marks the start of a sentence, SEP marks the end
def fixed_build_inputs(self, token_ids_0, token_ids_1=None):
    cls_id = self.cls_token_id # the "start" token
    sep_id = self.sep_token_id # the "end" token

    if token_ids_1 is None:
        # Single input: CLS + tokens + SEP
        return [cls_id] + token_ids_0 + [sep_id]

    # Two inputs: CLS + first + SEP + second + SEP
    return [cls_id] + token_ids_0 + [sep_id] + token_ids_1 + [sep_id]


# Apply the fix only if the method is not already defined in the tokenizer
if not hasattr(BertTokenizer, "build_inputs_with_special_tokens"):
    BertTokenizer.build_inputs_with_special_tokens = fixed_build_inputs

if not hasattr(BertTokenizerFast, "build_inputs_with_special_tokens"):
    BertTokenizerFast.build_inputs_with_special_tokens = fixed_build_inputs

## Step 7: Define Text Cleaning Functions

Before comparing answers, we clean the text so that small formatting differences
do not unfairly affect the scores.

We use two levels of cleaning:
- simple_clean: lowercase the text and remove extra spaces
- normalize_answer: also remove file names, special characters, and placeholder words

In [ ]:
# Basic cleaning: convert to lowercase and remove extra whitespace
def simple_clean(text):
    text = str(text).strip().lower()
    text = " ".join(text.split()) # collapse multiple spaces into one
    return text


# Deeper cleaning: also remove file names, symbols, and "nothing_found" placeholders
def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"\|.*", " ", text)

    # Remove file names that end in .txt (for example "document.txt")
    text = re.sub(r"\b[a-z0-9_\-]+\.txt\b", " ", text)

    # Remove the placeholder text used when no answer was found
    text = text.replace("nothing_found", " ")
    text = text.replace("nothing found", " ")

    # Keep only letters, numbers, German characters, and the section symbol
    text = re.sub(r"[^a-z0-9äöüß§ ]+", " ", text)

    # Remove extra spaces
    text = " ".join(text.split())
    return text

## Step 8: Define Fallback Detection and Source Matching Functions

Some model answers say "I could not find an answer in the context"
instead of giving a real answer. We call this a "fallback" response.

We also check whether the model correctly cited the relevant legal sources,
such as paragraph references like "§ 5 Abs. 1".

In [ ]:
# Check if a model answer is a fallback ("I could not answer")
def is_fallback(text):
    text = str(text).lower()
    text = text.replace("nothing_found", " nothing found ")

    # Remove special characters
    text = re.sub(r"[^a-z0-9äöüß ]+", " ", text)
    text = " ".join(text.split())

    # Check for common words used in fallback answers
    has_keine = "keine" in text or "kein" in text
    has_antwort = "antwort" in text
    has_kontext = "kontext" in text or "context" in text
    has_nothing = "nothing found" in text

    # It is a fallback if it says "nothing found" or "no answer in context"
    return has_nothing or (has_keine and has_antwort and has_kontext)


# Clean a legal reference string so it can be compared to others
def normalize_reference(text):
    text = str(text).lower()
    text = text.replace("abs.", "abs") # normalize abbreviation
    text = text.replace("lit.", "lit") # normalize abbreviation
    text = re.sub(r"\s+", " ", text) # collapse extra spaces
    text = re.sub(r"[^a-z0-9äöüß§ ]+", "", text) # remove punctuation
    return text.strip()


# Split a sources string into a list of individual references
# Example: "§ 5; § 7 Abs. 1" becomes ["§ 5", "§ 7 abs 1"]
def split_gold_sources(text):
    parts = re.split(r"[;,]", str(text)) # split at semicolons and commas
    refs = []
    for part in parts:
        part = normalize_reference(part)
        if part != "":
            refs.append(part)
    return refs


# Find all legal references mentioned inside a predicted answer
def extract_predicted_references(text):
    # This pattern matches things like "§ 5 Abs. 2 lit. a KEStG"
    matches = re.findall(
        r"§\s*\d+[a-zA-Z]?(?:\s*Abs\.?\s*\d+[a-zA-Z]?)?(?:\s*Z\s*\d+[a-zA-Z]?)?(?:\s*lit\.?\s*[a-zA-Z])?(?:\s+[A-Za-zÄÖÜäöüß]{2,}\s*\d{0,4})?",
        str(text),
        flags=re.IGNORECASE)

    refs = []
    for match in matches:
        match = normalize_reference(match)
        if match != "":
            refs.append(match)

    # Remove duplicates and sort alphabetically
    return sorted(list(set(refs)))

## Step 9: Compute All Scores

This is the main part of the notebook.

For every row in the CSV and for each of the three models, we:
1. Clean the predicted answer
2. Compute Exact Match, BLEU, ROUGE-1, ROUGE-L, and Normalized Similarity
3. Check whether the response is a fallback
4. Check how many correct legal sources the model mentioned
5. Assign an error type label

All results are stored in a list called "results".
At the end, we turn that list into a dataframe called "detailed_df".

In [ ]:
# This list will hold one dictionary per (question, model) pair
results = []

# Go through every row in the CSV one by one
for _, row in df.iterrows():

    current_id = str(row["id"])

    # Prepare the correct answer in two cleaned forms
    correct_raw = str(row["correct_answer"])
    correct_simple = simple_clean(correct_raw)
    correct_norm = normalize_answer(correct_raw)
    correct_tokens = correct_simple.split() # list of words, needed for BLEU

    # Get the list of correct legal sources for this question
    gold_refs = split_gold_sources(row["sources"])

    # Evaluate each of the three models for this row
    for model_column, model_name in models:

        predicted_raw = str(row[model_column])
        predicted_simple = simple_clean(predicted_raw)
        predicted_norm = normalize_answer(predicted_raw)
        predicted_tokens = predicted_simple.split() # list of words, needed for BLEU

        # Exact Match: 1 if both normalized texts are identical, 0 otherwise
        exact_match = int(predicted_norm == correct_norm and correct_norm != "")

        # BLEU: measures how many predicted words also appear in the correct answer
        bleu = 0.0
        if len(correct_tokens) > 0 and len(predicted_tokens) > 0:
            bleu = sentence_bleu([correct_tokens], predicted_tokens, smoothing_function=smoothing,)

        # ROUGE: measures text overlap from both directions
        rouge = scorer.score(correct_simple, predicted_simple)
        rouge1 = rouge["rouge1"].fmeasure # single-word overlap
        rougeL = rouge["rougeL"].fmeasure # longest sequence overlap

        # Normalized Similarity: a simple character-level similarity score (0 to 1)
        normalized_similarity = difflib.SequenceMatcher(
            None, correct_norm, predicted_norm).ratio()

        # Fallback Detection: did the model say it could not answer?
        fallback_detected = int(is_fallback(predicted_raw))

        # Reference Match: how many correct legal sources did the model cite?
        predicted_refs = extract_predicted_references(predicted_raw)
        reference_match = None # stays None if this question has no expected sources
        if len(gold_refs) > 0:
            matched = 0
            for gold_ref in gold_refs:
                for pred_ref in predicted_refs:
                    if gold_ref == pred_ref:
                        matched += 1
                        break # count each correct source only once
            reference_match = matched / len(gold_refs)

        # Error Type: describe what kind of mistake the model made (if any)
        error_type = "correct"
        if exact_match == 0:
            if fallback_detected == 1:
                error_type = "fallback"
            elif predicted_norm == "":
                error_type = "empty"
            elif normalized_similarity < 0.20:
                error_type = "low_similarity"
            else:
                error_type = "partial_match"

        # Save all scores for this (question, model) combination
        results.append({
            "id": current_id,
            "model": model_name,
            "correct_answer": correct_raw,
            "predicted_answer": predicted_raw,
            "correct_answer_normalized": correct_norm,
            "predicted_answer_normalized": predicted_norm,
            "gold_sources": " | ".join(gold_refs),
            "predicted_references": " | ".join(predicted_refs),
            "exact_match": exact_match,
            "bleu": round(bleu, 4),
            "rouge1": round(rouge1, 4),
            "rougeL": round(rougeL, 4),
            "normalized_similarity": round(normalized_similarity, 4),
            "fallback_detected": fallback_detected,
            "reference_match": reference_match,
            "error_type": error_type,
        })

# Convert the list of results into a dataframe
detailed_df = pd.DataFrame(results)

print("Evaluation loop done. Total rows:", len(detailed_df))
detailed_df.head()

## Step 10: Compute BERTScore

BERTScore uses a neural language model to measure how similar two answers are in meaning,
not just in wording.

For example, "the law applies" and "the regulation is in effect" would score low on BLEU
but high on BERTScore because the meaning is the same.

We use a multilingual BERT model because the data contains German text.

This step runs a few minutes due to running a language model.

In [ ]:
# Get all predicted and correct answer texts as plain Python lists
bert_predictions = detailed_df["predicted_answer_normalized"].tolist()
bert_references = detailed_df["correct_answer_normalized"].tolist()

# Compute BERTScore for all pairs at once
# Returns three scores: precision, recall, and F1: we only keep F1
_, _, bert_f1 = bert_score(
    bert_predictions,
    bert_references,
    model_type="bert-base-multilingual-cased", # supports German and English
    verbose=True, # show a progress bar while computing
)

# Add the F1 scores as a new column in the detailed dataframe
detailed_df["bert_f1"] = [round(s, 4) for s in bert_f1.tolist()]

print("BERTScore computed.")
detailed_df[["id", "model", "bert_f1"]].head(10)

## Step 11: Build the Summary Table

We group all results by model and compute the average score for each metric.
This produces one row per model, making it easy to compare all three side by side.

The table is sorted by BERTScore F1 so the best-performing model appears first.

In [ ]:
# Group by model and calculate the average for each score column
summary_df = detailed_df.groupby("model").agg(
    examples=("id", "count"),
    exact_match_rate=("exact_match", "mean"),
    bleu_avg=("bleu", "mean"),
    rouge1_avg=("rouge1", "mean"),
    rougeL_avg=("rougeL", "mean"),
    normalized_similarity_avg=("normalized_similarity", "mean"),
    bert_f1_avg=("bert_f1", "mean"),
    reference_match_avg=("reference_match", "mean"),
    fallback_rate=("fallback_detected", "mean"),
).reset_index()

# Convert the rate columns from decimals to percentages and round all values
summary_df["exact_match_rate"] = (summary_df["exact_match_rate"] * 100).round(2)
summary_df["bleu_avg"] = summary_df["bleu_avg"].round(4)
summary_df["rouge1_avg"] = summary_df["rouge1_avg"].round(4)
summary_df["rougeL_avg"] = summary_df["rougeL_avg"].round(4)
summary_df["normalized_similarity_avg"] = summary_df["normalized_similarity_avg"].round(4)
summary_df["bert_f1_avg"] = summary_df["bert_f1_avg"].round(4)
summary_df["reference_match_avg"] = summary_df["reference_match_avg"].round(4)
summary_df["fallback_rate"] = (summary_df["fallback_rate"] * 100).round(2)

# Sort by BERTScore F1 descending, with ROUGE-L as a tiebreaker
summary_df = summary_df.sort_values(by=["bert_f1_avg", "rougeL_avg"], ascending=False)

print("Summary table (best model first):")
summary_df

## Step 12: Analyze Errors

For every answer that was not an exact match, we count how many of each error type occurred.
This tells us not just how often a model is wrong, but in what way.

Error types:
- fallback: the model said it could not find an answer
- empty: the model returned no text at all
- low_similarity: the answer is very different from the correct one
- partial_match: the answer is somewhat close but not exactly correct

In [ ]:
# Keep only rows where the model did not produce an exact match
error_df = detailed_df[detailed_df["exact_match"] == 0]

# Count how many errors of each type appear for each model
error_summary_df = (error_df.groupby(["model", "error_type"]).size().reset_index(name="count"))

print("Error breakdown per model:")
error_summary_df

## Step 13: Save All Output Files

We save three CSV files to the output folder.

In [ ]:
# Save the full detailed results (one row per question per model)
detailed_df.to_csv(os.path.join(output_folder, "detailed_scores.csv"), index=False)

# Save the summary table (one row per model)
summary_df.to_csv(os.path.join(output_folder, "main_result_table.csv"), index=False)

# Save the error analysis
error_summary_df.to_csv(os.path.join(output_folder, "error_analysis.csv"), index=False)

## Summary

**What does this program do?**

It evaluates three AI models that answer legal questions in German.
It measures how accurate their answers are by comparing them against correct reference answers
using several scoring methods.

**How does it work, step by step?**

1. We read a CSV file containing questions, correct answers, and three model answers.
2. We clean the text to remove formatting differences that could unfairly affect scores.
3. For each answer, we compute five scores:
   - Exact Match: are the texts word-for-word identical after cleaning?
   - BLEU: how many words from the prediction also appear in the correct answer?
   - ROUGE-1 and ROUGE-L: how much text overlaps in either direction?
   - Normalized Similarity: how similar are the character sequences?
   - BERTScore: how similar is the meaning, even when different words are used?
4. We also check whether the model cited the correct legal paragraph references.
5. We group results by model, sort by BERTScore F1, and save everything to CSV files.
